# SH-wave modelling — IVF free surface, 3-layer model

Horizontally-layered medium with a flat free surface at z = 0 enforced via
the **Improved Vacuum Formulation** (IVF, Pan et al. 2018).

Units: **km/s**, **kg/m³**, **m**, **ms**.

| Parameter | Value |
|-----------|-------|
| Layer 1 | vs = 0.100 km/s, z = 0–2 m |
| Layer 2 | vs = 0.400 km/s, z = 2–130 m |
| Layer 3 | vs = 0.700 km/s, z ≥ 130 m |
| Density | 1970 kg/m³ (constant) |
| Grid spacing | 0.25 m |
| Domain | 300 × 150 m |
| Vacuum rows (IVF) | 3 rows above z = 0 |
| Source | Ricker 10 Hz, x = 150 m, z = 0 |
| Receivers | x = 100–200 m, Δx = 2 m, z = 0 |
| Max time | 850 ms |

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from devito import TimeFunction, NODE

from examples.seismic import AcquisitionGeometry
from examples.seismic.model import ModelSH
from examples.seismic.sh.operators import ForwardOperator

## 1. Model

In [ ]:
print(100/10/25)

In [ ]:
# ── Physical parameters ────────────────────────────────────────────────────────
vs1, vs2, vs3 = 0.100, 0.400, 0.700   # km/s
rho = 1.970                            # kg/m³

mu1 = np.float32(rho * vs1**2)   # 19.70
mu2 = np.float32(rho * vs2**2)   # 315.20
mu3 = np.float32(rho * vs3**2)   # 965.30
b_val = np.float32(1.0 / rho)    # buoyancy

print(f'μ₁={mu1:.2f}  μ₂={mu2:.2f}  μ₃={mu3:.2f}  kg/m³·(km/s)²')
print(f'b = {b_val:.4e}  m³/kg')

# ── Grid ───────────────────────────────────────────────────────────────────────
dx          = 0.4   # m
n_vac       = 3
Nx          = int(300 / dx) + 1      # 1201
Nz_sub      = int(150 / dx) + 1      # 601
nbl         = 100
space_order = 4

shape   = (Nx, Nz_sub + n_vac)
spacing = (dx, dx)
origin  = (0., -n_vac * dx)

# ── Layer z-indices in unpadded array ─────────────────────────────────────────
z1_idx = n_vac + int(round(2.5   / dx))   # first index of layer 2  (z=2 m)
z2_idx = n_vac + int(round(130.0 / dx))   # first index of layer 3  (z=130 m)

mu_arr = np.zeros(shape, dtype=np.float32)
b_arr  = np.zeros(shape, dtype=np.float32)
# vacuum rows (0..n_vac-1) stay zero
mu_arr[:, n_vac:z1_idx]  = mu1
mu_arr[:, z1_idx:z2_idx] = mu2
mu_arr[:, z2_idx:]       = mu3
b_arr[:, n_vac:]         = b_val

topo = np.full(Nx, n_vac, dtype=int)

model = ModelSH(
    origin=origin, spacing=spacing, shape=shape,
    space_order=space_order, mu=mu_arr, b=b_arr,
    nbl=nbl, topo=topo,
)

print(f'\ncritical_dt = {model.critical_dt:.4f} ms')
print(f'padded grid = {model.grid.shape}')
print(f'mu_z at vacuum boundary   = {model.mu_z.data[nbl + Nx//2, nbl + n_vac - 1]:.4f}  (expected 0)')
print(f'mu_z one row below        = {model.mu_z.data[nbl + Nx//2, nbl + n_vac]:.4f}       (expected {mu1:.2f})')

## 2. Geometry and time axis

In [ ]:
t0, tn = 0., 850.    # ms
f0 = 0.010            # kHz = 10 Hz

src_positions = np.array([[150., 0.]], dtype=np.float32)

rec_x = np.arange(50., 251., 2., dtype=np.float32)   # 51 receivers
rec_positions = np.column_stack(
    [rec_x, np.zeros_like(rec_x)]
).astype(np.float32)

geometry = AcquisitionGeometry(
    model, rec_positions, src_positions,
    t0=t0, tn=tn, src_type='Ricker', f0=f0,
)

dt = model.critical_dt
print(f'dt = {dt} ms,  nt = {geometry.nt}')

## 3. Operator and wavefield buffers

In [ ]:
op = ForwardOperator(model, geometry, space_order=space_order)

x, z = model.grid.dimensions

v = TimeFunction(name='v', grid=model.grid, space_order=space_order,
                 time_order=1, staggered=NODE)
tau_xy = TimeFunction(name='tau_xy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(x,))
tau_zy = TimeFunction(name='tau_zy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(z,))
rec = geometry.new_rec(name='rec')

## 4. Run

In [ ]:
op.apply(
    v=v, tau_xy=tau_xy, tau_zy=tau_zy,
    src=geometry.src, rec=rec,
    dt=dt, **model.physical_params(),
)
print(f'rec.data shape: {rec.data.shape},  |rec|_max = {np.abs(rec.data).max():.4e}')

## 5. Seismogram

Vertical-component particle velocity recorded at the free surface (z = 0).  
Colour is clipped symmetrically at the 98th percentile amplitude.

In [ ]:
seismogram = np.array(rec.data)
clip = np.percentile(np.abs(seismogram), 85) or 1.0

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(
    seismogram,
    aspect='auto',
    cmap='gray',
    vmin=-clip, vmax=clip,
    extent=[rec_x[0], rec_x[-1], tn, t0],
    interpolation='bicubic',
)
# plt.colorbar(im, ax=ax, label='v  [km/s · arb.]', shrink=0.85)
ax.set_xlabel('Receiver x [m]')
ax.set_ylabel('Time [ms]')
# ax.set_title('SH seismogram — IVF 3-layer model (10 Hz Ricker)')
plt.tight_layout()
plt.show()